# 🏥 CareFlow — Notebook 11: Statistical Process Control (SPC)
## Healthcare Quality Improvement, Shewhart $X-mR$ Charts & Nelson Rules

**Author:** CareFlow Quality & Patient Safety Team  
**Standards:** Institute for Healthcare Improvement (IHI), NHS England Statistical Process Control Guidelines  
**Dataset:** `careflow_analytics/output/fct_patient_journeys.parquet`

---

### Objectives:
1. Aggregate daily average Length of Stay (LOS) across 90 days.
2. Construct Shewhart Individuals and Moving Range ($X-mR$) charts with 3-Sigma limits ($UCL = \mu + 3\sigma, LCL = \mu - 3\sigma$).
3. Implement **Nelson Rules 1 through 4** to detect special-cause variation (systemic operational shocks).
4. Build a p-chart for daily proportion of SLA breaches.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

fct_df = pd.read_parquet('../careflow_analytics/output/fct_patient_journeys.parquet')
fct_df['visit_date'] = fct_df['visit_start_time'].dt.date

daily_los = fct_df.groupby('visit_date')['total_los_minutes'].agg(['mean', 'count']).reset_index()
daily_los.columns = ['date', 'mean_los', 'patient_volume']

print(f"Aggregated {len(daily_los)} daily clinical operational dates.")
daily_los.head()


In [ ]:
# 1. Shewhart Control Limits Calculation (Moving Range Method)
x = daily_los['mean_los'].values
mr = np.abs(np.diff(x))
mr_bar = np.mean(mr)
d2 = 1.128  # Constant for sample size n=2
sigma = mr_bar / d2

x_bar = np.mean(x)
ucl = x_bar + 3 * sigma
lcl = max(0, x_bar - 3 * sigma)
uwl = x_bar + 2 * sigma
lwl = max(0, x_bar - 2 * sigma)

print(f"Center Line (Mean LOS): {x_bar:.2f} min")
print(f"Upper Control Limit (UCL): {ucl:.2f} min")
print(f"Lower Control Limit (LCL): {lcl:.2f} min")
print(f"Estimated Process Sigma: {sigma:.2f} min")


In [ ]:
# 2. Nelson Rules Special Cause Violation Detection
# Rule 1: Point outside 3-sigma (UCL or LCL)
r1_violations = (x > ucl) | (x < lcl)

# Rule 2: 9 consecutive points on same side of center line
r2_violations = np.zeros(len(x), dtype=bool)
for i in range(8, len(x)):
    if np.all(x[i-8:i+1] > x_bar) or np.all(x[i-8:i+1] < x_bar):
        r2_violations[i] = True

# Rule 3: 6 consecutive points steadily increasing or decreasing
r3_violations = np.zeros(len(x), dtype=bool)
for i in range(5, len(x)):
    diffs = np.diff(x[i-5:i+1])
    if np.all(diffs > 0) or np.all(diffs < 0):
        r3_violations[i] = True

print(f"Nelson Rule 1 Violations (Outliers > 3σ): {r1_violations.sum()}")
print(f"Nelson Rule 2 Violations (Shifts > 9 pts):  {r2_violations.sum()}")
print(f"Nelson Rule 3 Violations (Trends > 6 pts): {r3_violations.sum()}")


In [ ]:
# 3. High-Resolution Healthcare SPC Chart
plt.figure(figsize=(14, 6))

plt.plot(daily_los['date'], x, color='#06b6d4', marker='o', markersize=4, linewidth=1.5, label='Daily Mean LOS')
plt.axhline(x_bar, color='#10b981', linestyle='-', linewidth=2, label=f'Center Line ({x_bar:.1f}m)')
plt.axhline(ucl, color='#f43f5e', linestyle='--', linewidth=2, label=f'UCL (+3σ = {ucl:.1f}m)')
plt.axhline(lcl, color='#f43f5e', linestyle='--', linewidth=2, label=f'LCL (-3σ = {lcl:.1f}m)')
plt.axhline(uwl, color='#f59e0b', linestyle=':', alpha=0.6, label='Warning Limit (±2σ)')
plt.axhline(lwl, color='#f59e0b', linestyle=':', alpha=0.6)

# Highlight special-cause violations in red
viol_idx = np.where(r1_violations)[0]
plt.scatter(daily_los['date'].iloc[viol_idx], x[viol_idx], color='#f43f5e', s=90, zorder=5, label='Special Cause Breach (Rule 1)')

plt.title('Healthcare SPC X-mR Control Chart: Emergency Department Length of Stay')
plt.xlabel('Date')
plt.ylabel('Mean LOS (Minutes)')
plt.xticks(rotation=30)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()
